# EWMA-Toleranzintervall-Pairtrading über Cliquen (Lin, Chen & Syu 2021)

**Was dieses Notebook tut:** simuliert die Pairtrading-Strategie aus Lin, Chen & Syu (2021,
"Multi-asset pair-trading strategy: A statistical learning approach") auf den Mitgliedern der
in `02_tda_clustering` gefundenen und in `data/processed/cliques/` gecachten Cliquen, und
optimiert die Signal-Parameter dieser Strategie automatisiert mit Optuna.

**Methodischer Unterschied zu einer klassischen RSI/Bollinger-Engine:** statt einzelne Assets
gegen ihre eigenen Bänder zu handeln, wird hier für JEDES Paar innerhalb einer Clique die
Log-Return-Differenz `Rt = r_Ai,t - r_Aj,t` gebildet, per EWMA-Volatilität normiert
(`Xt = Rt / sigma_t`), und über ein nichtparametrisches Wilks-Toleranzintervall (statt eines
willkürlichen Z-Score-Schwellenwerts) auf Signifikanz geprüft — datengetrieben statt
willkürlich, siehe Additional-Info-PDF.

**Wichtige Abgrenzung** (nicht mit dem Preis-Level-Spread aus der ADF/KPSS/Hurst-
Stationaritätsdiagnostik in `03_signal_generation/TradeData.ipynb` verwechseln): dort wurde die
Renditedifferenz durch einen Preis-Level-Spread ersetzt, weil DAS der richtige Prozess für
klassische Stationaritätstests ist. HIER, für die Lin-et-al.-Signalgenerierung, ist die
Renditedifferenz selbst die vom Paper vorgeschriebene, korrekte Größe — sie bleibt unverändert.

**Aufbau dieses Notebooks (bewusst schlank gehalten — EIN Tuning-Weg, kein Nebeneinander
mehrerer Varianten):**
1. Daten & Cliquen-Cache laden.
2. Kurze, formelfreie Erklärung, wie die Handelssignale überhaupt entstehen (Abschnitt 4) —
   Grundlage, um die Parameter im Optuna-Suchraum (Abschnitt 5) inhaltlich einordnen zu können.
3. Optuna-Tuning (Abschnitt 5): zweistufige Hyperparameter-Suche (günstiger Signal-Screen als
   Vorfilter + voller Walk-Forward-Backtest) mit persistentem SQLite-Storage und Live-Dashboard.
4. Einmalige finale Auswertung auf einem während des gesamten Tunings unberührten Testzeitraum
   (Abschnitt 5b) — DAS ist die Zahl, die als Out-of-Sample-Ergebnis zitiert werden sollte.

Die eigentliche Logik (EWMA, Spreads, Wilks-Toleranzintervall, Signal-/Trade-Simulation,
Equity-Kurve, Kennzahlen, Plots, Walk-Forward-Fenster) steckt komplett in
`pairtrading_engine.py`; die Tuning-Logik (Optuna, zweistufiges Pruning, Zielgröße,
persistenter Storage) in `tune_backtest.py`. Dieses Notebook ist nur das Interface darüber —
es enthält selbst keine Backtest- oder Optimierungslogik. Die Kern-Funktion
`wilks_tolerance_limits` ist in `test_pairtrading_engine.py` gegen die zugrundeliegende
Beta-Verteilungsformel abgesichert (nicht gegen einen unsicheren Tabellenwert) — vor jeder
produktiven Nutzung sollten diese Tests grün sein (`python test_pairtrading_engine.py`).


In [13]:
# =========================
# 1. Setup & Imports
# =========================
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str(Path("../02_tda_clustering").resolve()))

import ipywidgets as widgets
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

import pairtrading_engine as pte
from config import (
    ASSET_DAILY_PATH,
    ASSET_ID_COL,
    ASSETS_PATH,
    CLOSE_COL,
    MARKET_SYMBOL_COL,
    OHLCV_DIR,
    PROCESSED_DIR,
    PROJECT_ROOT,
    TIMESTAMP_COL,
)
from helperfunctions import STABLECOIN_SYMBOLS

print("Imports OK")

Imports OK


# 1. Daten laden

Lädt die 5-Minuten-Kerzen-OHLCV-Daten aus `data/processed/ohlcv/` und mappt sie über die
Asset-Stammdaten (`assets.parquet`) auf `asset_id` — dieselbe ID, mit der auch die Cliquen im
nächsten Abschnitt referenziert werden. Stablecoins werden ausgeschlossen (`STABLECOIN_SYMBOLS`
aus `helperfunctions.py`), da Pairtrading zwischen zwei Stablecoins (beide ~konstant bei 1 USD)
kein sinnvolles Signal liefern kann — die Renditedifferenz wäre strukturell nahe Null,
unabhängig von jedem Marktereignis.


In [14]:
assets_df = pd.read_parquet(ASSETS_PATH)[["asset_id", "symbol"]].drop_duplicates()
id_to_symbol = dict(zip(assets_df["asset_id"], assets_df["symbol"]))

excluded_ids_stablecoins: set[int] = set(
    assets_df.loc[assets_df["symbol"].isin(STABLECOIN_SYMBOLS), "asset_id"].tolist()
)

ohlcv_files = sorted(OHLCV_DIR.glob("*.parquet"))
raw = pd.concat([pd.read_parquet(f) for f in ohlcv_files], ignore_index=True)
raw["base_symbol"] = raw[MARKET_SYMBOL_COL].str.replace("USDT$", "", regex=True)
raw = raw.merge(assets_df.rename(columns={"symbol": "base_symbol"}), on="base_symbol", how="left")
raw = raw.dropna(subset=["asset_id"])
raw["asset_id"] = raw["asset_id"].astype(int)
raw[TIMESTAMP_COL] = pd.to_datetime(raw[TIMESTAMP_COL], utc=True)
raw = raw.loc[~raw["asset_id"].isin(excluded_ids_stablecoins)]
ohlcv_df = raw[[TIMESTAMP_COL, ASSET_ID_COL, CLOSE_COL]].copy()

data_min = ohlcv_df[TIMESTAMP_COL].min().date()
data_max = ohlcv_df[TIMESTAMP_COL].max().date()

print(f"Zeitraum: {data_min} → {data_max}")
print(f"OHLCV: {ohlcv_df.shape[0]:,} Zeilen | Assets: {ohlcv_df[ASSET_ID_COL].nunique()}")

Zeitraum: 2024-07-23 → 2026-07-22
OHLCV: 8,553,834 Zeilen | Assets: 45


# 2. Cliquen-Cache laden

Lädt die von der TDA-Pipeline (`02_tda_clustering`) gefundenen Cliquen aus dem Cache in
`data/processed/cliques/` — **keine eigene Clique-Discovery hier**. Eine Clique ist eine Gruppe
von Assets, die an einem bestimmten Tag (`discovery_date`) laut Vietoris-Rips-Persistenzanalyse
eng topologisch zusammenhängen (siehe Abschnitt 4 für die intuitive Erklärung, warum das für
Pairtrading relevant ist). Zwei Varianten stehen zur Verfügung:
- `raw`: Cliquen direkt auf den Rohreturns.
- `residual`: Cliquen auf Residual-Returns (nach Herausrechnen eines Marktfaktors) — reduziert
  das Risiko, dass eine Clique nur den allgemeinen Marktauf-/abschwung abbildet statt einer
  echten paarweisen Kointegrations-/Zusammenhangsstruktur.

Welche Variante fürs Tuning verwendet wird, ist in Abschnitt 5 (Optuna-Widgets) einstellbar.


In [15]:
CLIQUES_DIR = PROCESSED_DIR / "cliques"


def _load_cliques_cache(variant: str) -> pd.DataFrame:
    path = CLIQUES_DIR / f"cliques_{variant}.parquet"
    if not path.exists():
        raise FileNotFoundError(
            f"{path.relative_to(PROJECT_ROOT)} nicht gefunden -- zuerst "
            "02_tda_clustering/05_clique_cache.ipynb laufen lassen."
        )
    df = pd.read_parquet(path)
    df["discovery_date"] = pd.to_datetime(df["discovery_date"], utc=True)
    df["asset_ids"] = df["asset_ids"].apply(list)  # Parquet gibt Listen als np.ndarray zurück
    return df


cliques_by_variant = {
    "raw": _load_cliques_cache("raw"), #raw bedeutet, dass die Cliquen direkt aus den TDA-Ergebnissen stammen, ohne weitere Filterung oder Anpassung.
    "residual": _load_cliques_cache("residual"), #residual bedeutet, dass die Cliquen aus den TDA-Ergebnissen stammen, aber auf Basis der Residualwerte berechnet wurden, um die Auswirkungen von Trends oder saisonalen Mustern zu reduzieren.
}
basket_labels_available = sorted(cliques_by_variant["raw"]["basket"].unique())

for variant, df in cliques_by_variant.items():
    print(f"cliques_{variant}: {len(df)} Cliquen über {df['discovery_date'].nunique()} Tage "
          f"({df['discovery_date'].min().date()} → {df['discovery_date'].max().date()})")

cliques_raw: 1750 Cliquen über 674 Tage (2024-07-29 → 2026-07-22)
cliques_residual: 1188 Cliquen über 596 Tage (2024-07-29 → 2026-07-22)


# 3. Wie entstehen die Trading-Signale? (einfach erklärt)

Bevor es an das Parameter-Tuning geht, kurz und ohne Formeln: **was steckt eigentlich hinter
den Parametern, die Optuna im nächsten Abschnitt optimiert?**

**1. Eine Clique ist eine Gruppe von Coins, die sich normalerweise ähnlich bewegen.**
Diese Gruppen wurden vorher mit Topologischer Datenanalyse (TDA) gefunden — Coins landen
nur dann zusammen in einer Clique, wenn ihre Kursverläufe über die Zeit eng zusammenhängen.

**2. Für jedes Coin-Paar in der Clique wird laufend die Differenz ihrer Renditen berechnet.**
Bewegen sich zwei Coins normalerweise im Gleichschritt, ist diese Differenz meistens nahe
Null. Wird sie plötzlich groß, bedeutet das: die beiden Coins haben sich kurzfristig
"auseinanderbewegt" — einer ist stärker gestiegen oder gefallen als der andere, obwohl sie
sich eigentlich ähnlich verhalten sollten.

**3. "Groß" wird nicht willkürlich festgelegt, sondern aus der jüngeren Vergangenheit gelernt.**
Für jedes Paar schaut sich die Strategie ein Zeitfenster der letzten Kerzen an (`lookback_candles`
— **einer der Optuna-Suchparameter**) und berechnet daraus eine statistische Toleranzgrenze:
einen Bereich, in dem sich die Renditedifferenz normalerweise bewegt. `p_content` (**ebenfalls
tunbar**) legt fest, wie viel Prozent der historischen Werte innerhalb dieses Bereichs liegen
sollen (z. B. 95 %), `confidence` (fest auf 0.95, siehe Abschnitt 5) sagt, wie sicher man sich
bei dieser Schätzung sein will. Vorteil gegenüber einem festen Schwellenwert: die Grenze passt
sich automatisch an, ob ein Coin gerade ruhig oder volatil ist. Ruhige Phasen -> engere Grenze,
turbulente Phasen -> weitere Grenze.

**4. Verlässt die Renditedifferenz eines Paares diese Toleranzgrenze, entsteht ein Signal.**
Das interpretiert die Strategie als: "diese beiden Coins sind kurzfristig auseinandergelaufen,
das sollte sich wieder normalisieren." Der Trade wettet auf genau diese Rückkehr zum
Normalzustand (Mean-Reversion) — man kauft den relativ zurückgebliebenen Coin (Long) und
verkauft gleichzeitig den relativ vorgelaufenen Coin (Short), sodass nur die *Differenz*
zwischen den beiden zählt, nicht die allgemeine Marktrichtung. `trade_mode` (**tunbar**)
entscheidet, ob dabei NUR das Paar mit dem stärksten Ausschlag pro Zeitpunkt gehandelt wird
(`max_only`, repliziert Lin et al. exakt) oder ALLE Paare, die ihre Grenze verletzen
(`all_significant`).

**5. Der Trade wird wieder geschlossen, sobald sich die Differenz normalisiert hat**
(zurück innerhalb der Toleranzgrenze), oder spätestens nach der eingestellten maximalen
Haltedauer (`max_holding_hours`, **tunbar**), damit keine Position unbegrenzt offenbleibt.
Wie oft die Toleranzgrenze neu geschätzt wird (statt sie über lange Zeit fest zu halten),
steuert `recalib_every` (**tunbar**) — und `max_concurrent_positions` (**tunbar**) begrenzt,
wie viele Positionen gleichzeitig offen sein dürfen (Kapitalallokations-/Risikogrenze).

**Warum das Ganze funktionieren könnte:** die Coins in einer Clique wurden gerade *weil*
sie eng zusammenhängen ausgewählt — kurzfristige Abweichungen von diesem normalen
Zusammenhang gelten als vorübergehende "Übertreibung", die sich statistisch häufig wieder
zurückbildet. Ob und wie stark das tatsächlich zutrifft, hängt genau von den sechs Parametern
ab, die Abschnitt 5 mit Optuna durchsucht — deshalb ist die Signal-Erklärung hier direkt die
Grundlage, um die dortigen Suchraum-Grenzen und Ergebnisse einzuordnen.

**Was NICHT hier optimiert wird** (siehe Abschnitt 5 für die ausführliche Begründung): OB eine
Clique/ein Paar überhaupt als handelbar gilt, wird bereits VOR diesem Notebook entschieden —
durch die TDA-Persistenzschwelle (`min_persistence`, Clique-Discovery) und durch statistische
Stationaritätstests (ADF/KPSS/FDR, `03_signal_generation/TradeData.ipynb`). Diese beiden
vorgelagerten Filter sind theoriegeleitet und bewusst NICHT Teil der Optuna-Suche.


# 5. Optuna-Tuning (zweistufig, walk-forward, persistenter Storage + Dashboard)

Der EINZIGE Tuning-Weg in diesem Notebook (bewusst so reduziert -- kein Nebeneinander
mehrerer Varianten): nutzt [Optuna](https://optuna.org/) (TPE-Sampler) statt Grid-Search,
UND einen zweistufigen Ansatz, der teure volle Backtests vermeidet, wo ein guenstiger
Signal-Screen bereits erkennen laesst, dass eine Parameterkombination nicht
vielversprechend ist:

- **Stufe 1 (`pairtrading_engine.screen_signal_quality`):** berechnet NUR die Signalkette
  (Spreads -> EWMA-Normierung -> Xmax -> Toleranzgrenzen), OHNE Trades zu simulieren, und
  daraus einen Information Coefficient (IC) zwischen Signalstaerke und nachfolgender
  Spread-Rueckkehr. Reicht der IC nicht (siehe `tune_backtest.SCREEN_MIN_IC`), wird der
  Optuna-Trial per `optuna.TrialPruned()` abgebrochen -- **kein** voller Backtest noetig.
- **Stufe 2 (`pairtrading_engine.run_pairtrading_backtest`):** nur fuer Trials, die Stufe 1
  bestehen, laeuft der volle Backtest -- ueber ALLE Walk-Forward-Validierungsfenster, nicht
  nur ueber einen einzelnen Zeitraum. Zielgroesse: mittlere Validierungs-Rendite ueber die
  Fenster MINUS ein Strafterm fuer hohe Streuung zwischen den Fenstern (Robustheit statt
  Glueckstreffer, siehe `tune_backtest._objective_metric`).

**Zeitliche Trennung (kritisch, siehe `pairtrading_engine.generate_walkforward_windows`):**
der Gesamtzeitraum wird in drei Bloecke zerlegt:

1. **Trainings-/Validierungszeitraum**, in dem Optuna sucht (walk-forward: mehrere
   aufeinanderfolgende Train/Val-Fenster).
2. Ein **Embargo-Puffer** zwischen Validierung und finalem Test, um Leakage aus
   ueberlappenden Rekalibrierungsfenstern zu vermeiden.
3. Ein **finaler Testzeitraum**, der waehrend des GESAMTEN Tunings unberuehrt bleibt und erst
   am Ende, EINMAL, mit den final gewaehlten Parametern durchlaufen wird (letzte Zelle dieses
   Abschnitts). `pairtrading_engine.assert_no_test_period_leakage` prueft dies vor JEDEM
   einzelnen Trial und bricht mit einer klaren Fehlermeldung ab, falls der Suchraum den
   Testzeitraum beruehrt.

**Widget-Bedienung dieser drei Bloecke:** um eine widerspruechliche manuelle Datumsauswahl
(z.B. Test-Zeitraum VOR Train/Val, oder ueberlappende Zeitraeume) strukturell auszuschliessen,
werden Train/Val-Zeitraum und Test-Zeitraum NICHT mehr einzeln eingegeben. Stattdessen waehlt
man nur EINEN Gesamtzeitraum (Gesamt-Start/Gesamt-Ende), einen Testanteil in Prozent (welcher
Anteil des Gesamtzeitraums, gerechnet vom ENDE her, als Test reserviert wird) und den
Embargo-Puffer in Tagen -- die vier tatsaechlich verwendeten Datumswerte (Train/Val
Start/Ende, Test Start/Ende) werden daraus automatisch berechnet und nur noch (read-only) zur
Kontrolle angezeigt.

**Theoriegeleitete Filter vs. tunbare Signalparameter** (methodische Nachvollziehbarkeit,
siehe Ernie Chan, Kapitel Backtesting, zu Data-Snooping-Bias -- ausfuehrliche Begruendung im
Docstring von `tune_backtest.py`):
- **NICHT** Teil der Optuna-Suche: `min_persistence` (H0-Persistenzschwelle der
  Vietoris-Rips-Clique-Discovery, `02_tda_clustering/helperfunctions.py`) sowie ADF-autolag,
  KPSS-regression, FDR-alpha (Stationaritaets-/Mean-Reversion-Diagnostik auf
  Preis-Level-Spreads, `03_signal_generation/TradeData.ipynb`) -- diese bestimmen, WELCHE
  Cliquen/Paare ueberhaupt als handelbar gelten, BEVOR irgendein Backtest beginnt. Es sind
  literaturbasierte statistische Vortests mit klaren Konventionen, keine Hyperparameter einer
  Handelsstrategie -- sie ex-post gegen die Sharpe Ratio zu optimieren waere klassisches
  Data-Snooping.
- **Teil** der Optuna-Suche: `lookback_candles`, `p_content`, `trade_mode`, `recalib_every`,
  `max_holding_hours`, `max_concurrent_positions` -- reine Signal-/Trade-Management-Parameter
  INNERHALB von `run_pairtrading_backtest`, nachdem die Cliquen/Paare bereits feststehen.

**Persistenter SQLite-Storage + optuna-dashboard:** standardmaessig aktiviert (Checkbox
"Persistenter SQLite-Storage" unten). Die Study wird in einer SQLite-Datei
(`tune_backtest.DEFAULT_STORAGE_PATH`, per Widget aenderbar) gespeichert, ueberlebt
Notebook-/Kernel-Neustarts (`load_if_exists=True` -- ein erneuter Lauf mit demselben
Study-Namen haengt weitere Trials an, statt bei 0 neu zu beginnen), und kann WAEHREND das
Tuning laeuft parallel von `optuna-dashboard` gelesen werden:

```
optuna-dashboard sqlite:///<Pfad zur .db-Datei, siehe SQLite-Datei-Feld unten> --port 8080 --server wsgiref
```

**Windows-Hinweis zum Dashboard-Server:** `--server wsgiref` ist noetig, weil der Optuna-Dashboard-Default (`gunicorn`) auf Windows mit `ModuleNotFoundError: No module named 'fcntl'` fehlschlaegt (gunicorn ist Unix-only) -- getestet und bestaetigt, `wsgiref` (Python-Standardbibliothek, plattformunabhaengig) funktioniert fehlerfrei als Ersatz.

Dann im Browser: **http://127.0.0.1:8080** -- zeigt Parameter-Importance, Parallel-Coordinate-
Plot und Optimization-History live, aktualisiert sich automatisch waehrend neue Trials
abgeschlossen werden. Mehrere Studies (z.B. je Basket) landen in derselben `.db`-Datei und
werden im Dashboard einzeln zur Auswahl gelistet.

**Windows-Hinweis:** `tune_backtest.py` setzt beim Import `KMP_DUPLICATE_LIB_OK=TRUE`
(bekanntes OpenMP/DLL-Problem bei mehreren numpy/scipy-Runtimes im selben Prozessbaum,
relevant falls `study.optimize(..., n_jobs>1)` fuer parallele Trials genutzt wird -- hier
nicht standardmaessig aktiv, aber vom Aufrufer waehlbar).

Ruft ausschliesslich bereits vorhandene Bausteine aus `pairtrading_engine.py` auf
(`run_pairtrading_backtest` wird NICHT veraendert, nur wiederverwendet) -- die eigentliche
Tuning-Logik liegt in `tune_backtest.py`.


In [16]:
# =====================================================================
# Abschnitt 5: Widgets fuer das Optuna-Tuning
# =====================================================================
# tune_backtest.py enthaelt die gesamte Tuning-Logik (Suchraum, Zielgroesse,
# Pruning, persistenter Storage) -- dieses Notebook baut nur die Bedienober-
# flaeche darueber und ruft tb.tune_pairtrading_strategy() im Callback
# (naechste Zelle) auf. Siehe Modul-Docstring von tune_backtest.py fuer die
# vollstaendige Begruendung aller Design-Entscheidungen.
import tune_backtest as tb

# --- Universum: welche Cliquen-Variante und welcher Basket wird getunt? ---
# "raw" vs. "residual" siehe Abschnitt 2 (Cliquen-Cache laden).
ot_w_variant = widgets.Dropdown(options=["raw", "residual"], value="raw", description="Cliquen-Variante:", style={"description_width": "160px"})
ot_w_basket = widgets.Dropdown(options=basket_labels_available, value=basket_labels_available[0], description="Basket:", style={"description_width": "160px"})

# --- Zeitliche Trennung: EIN Gesamtzeitraum + Testanteil (%) + Embargo ---
# Statt vier freien Datumsfeldern (Train/Val Start/Ende, Test Start/Ende),
# die widerspruechliche Kombinationen zulassen wuerden (z.B. Test VOR
# Train/Val, oder ueberlappende Zeitraeume), wird hier NUR EIN Gesamt-
# zeitraum gewaehlt und Train/Val/Test WERDEN DARAUS BERECHNET:
#   [Gesamt-Start, Gesamt-Ende)
#     = [Train/Val-Block][Embargo-Puffer][Test-Block, am Ende, test_pct %]
# Das macht eine falsche Datumsauswahl strukturell unmoeglich -- es gibt
# keine Freiheitsgrade mehr, die man widerspruechlich einstellen koennte.
# Die drei Bloecke aus dem Modul-Docstring von tune_backtest.py bleiben
# inhaltlich identisch:
# 1) [Train/Val Start, Train/Val Ende): der Bereich, INNERHALB dessen Optuna
#    Walk-Forward-Fenster bildet und sucht (siehe Walk-Forward-Widgets unten).
# 2) Embargo-Puffer (Tage) zwischen Train/Val Ende und Test Start: verhindert,
#    dass Rekalibrierungs-Lookback-Fenster des finalen Tests in den Trainings-
#    zeitraum hineinreichen (Leakage-Schutz).
# 3) [Test Start, Test Ende): bleibt WAEHREND DES GESAMTEN TUNINGS unberuehrt
#    und wird erst in Abschnitt 5b (ganz am Ende, EINMAL) verwendet.
# pairtrading_engine.assert_no_test_period_leakage prueft zusaetzlich vor
# JEDEM Trial, dass kein Walk-Forward-Fenster in [Test Start, Test Ende)
# hineinreicht -- doppelte Absicherung on top der strukturellen Loesung hier.
ot_w_overall_start = widgets.DatePicker(description="Gesamt-Start:", style={"description_width": "160px"})
ot_w_overall_end = widgets.DatePicker(description="Gesamt-Ende:", style={"description_width": "160px"})
ot_w_test_pct = widgets.IntSlider(value=15, min=5, max=50, description="Testanteil (%):", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))
ot_w_embargo_days = widgets.IntSlider(value=5, min=0, max=30, description="Embargo (Tage):", style={"description_width": "160px"}, layout=widgets.Layout(width="420px"))

# Read-only Anzeige der DARAUS berechneten Zeitraeume -- nicht editierbar,
# nur zur Kontrolle, was tatsaechlich verwendet wird.
ot_w_train_val_start = widgets.DatePicker(description="-> Train/Val Start:", disabled=True, style={"description_width": "160px"})
ot_w_train_val_end = widgets.DatePicker(description="-> Train/Val Ende:", disabled=True, style={"description_width": "160px"})
ot_w_test_start = widgets.DatePicker(description="-> Test Start:", disabled=True, style={"description_width": "160px"})
ot_w_test_end = widgets.DatePicker(description="-> Test Ende:", disabled=True, style={"description_width": "160px"})


def _ot_default_overall_range():
    """Default-Gesamtzeitraum: Januar-Februar 2025 (2 Monate) -- klein
    gehalten, damit ein Testlauf schnell durchlaeuft (siehe Chat-Diskussion
    zu Ladezeiten). Faellt auf [data_min, data_max] zurueck, falls dieser
    Bereich ausserhalb der geladenen Daten liegt."""
    fixed_start = pd.Timestamp("2025-01-01")
    fixed_end = pd.Timestamp("2025-03-01")
    data_min_ts = pd.Timestamp(data_min)
    data_max_ts = pd.Timestamp(data_max)
    if fixed_start >= data_min_ts and fixed_end <= data_max_ts:
        return fixed_start.date(), fixed_end.date()
    return data_min, data_max


_ot_os, _ot_oe = _ot_default_overall_range()
ot_w_overall_start.value = _ot_os
ot_w_overall_end.value = _ot_oe


def _ot_recompute_split(change=None):
    """Berechnet Train/Val- und Test-Zeitraum aus Gesamtzeitraum, Testanteil
    (%) und Embargo (Tage) neu -- bei JEDER Aenderung eines der drei Eingabe-
    Widgets (siehe .observe()-Aufrufe unten). Rechnet mit pd.Timestamp statt
    datetime.date, weil `date + pd.Timedelta` einen date (kein Timestamp)
    zurueckgibt und somit KEIN .date() kennt -- erst am Ende wird auf
    .date() konvertiert."""
    if ot_w_overall_start.value is None or ot_w_overall_end.value is None:
        return
    overall_start = pd.Timestamp(ot_w_overall_start.value)
    overall_end = pd.Timestamp(ot_w_overall_end.value)
    if overall_end <= overall_start:
        return

    total_days = (overall_end - overall_start).days
    test_days = max(1, int(round(total_days * ot_w_test_pct.value / 100)))
    embargo_days = ot_w_embargo_days.value
    test_end = overall_end

    # Train/Val bekommt den Rest, NACHDEM Test-Anteil und Embargo abgezogen
    # sind -- garantiert train_val_end + embargo <= test_start, keine
    # Ueberlappung moeglich.
    train_val_start = overall_start
    train_val_end = overall_end - pd.Timedelta(days=test_days + embargo_days)
    if train_val_end <= train_val_start:
        # Testanteil + Embargo zu gross fuer den Gesamtzeitraum -- Train/Val
        # auf ein Minimum von 1 Tag klemmen statt negativ/leer werden zu
        # lassen (verhindert eine stille Fehlkonfiguration).
        train_val_end = train_val_start + pd.Timedelta(days=1)
    test_start = train_val_end + pd.Timedelta(days=embargo_days)
    if test_start > test_end:
        # Selbst nach dem Klemmen von train_val_end reicht der verbleibende
        # Platz nicht fuer den vollen Embargo-Puffer VOR test_end -- test_start
        # zusaetzlich auf test_end klemmen, damit NIE test_start > test_end
        # entstehen kann (sonst waere der Testzeitraum negativ/undefiniert).
        test_start = test_end

    ot_w_train_val_start.value = train_val_start.date()
    ot_w_train_val_end.value = train_val_end.date()
    ot_w_test_start.value = test_start.date()
    ot_w_test_end.value = test_end.date()


ot_w_overall_start.observe(_ot_recompute_split, names="value")
ot_w_overall_end.observe(_ot_recompute_split, names="value")
ot_w_test_pct.observe(_ot_recompute_split, names="value")
ot_w_embargo_days.observe(_ot_recompute_split, names="value")
_ot_recompute_split()

# --- Walk-Forward-Fenster INNERHALB von [Train/Val Start, Train/Val Ende) ---
# pairtrading_engine.generate_walkforward_windows baut daraus eine Liste
# aufeinanderfolgender (train, val)-Bloecke: train_days Training, dann
# wf_embargo_days Puffer, dann val_days Validierung, dann step_days weiter.
# Jeder Optuna-Trial wird (in Stufe 2, siehe ot-callback) ueber ALLE diese
# Validierungsfenster ausgewertet, nicht nur ueber eines -- das ist die
# Walk-Forward-Robustheitspruefung, siehe tune_backtest._objective_metric.
ot_w_train_days = widgets.IntSlider(value=30, min=5, max=90, description="Trainingsfenster (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
ot_w_val_days = widgets.IntSlider(value=10, min=3, max=30, description="Validierungsfenster (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
ot_w_step_days = widgets.IntSlider(value=15, min=3, max=60, description="Schrittweite (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
ot_w_wf_embargo_days = widgets.IntSlider(value=2, min=0, max=20, description="WF-internes Embargo (Tage):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))

# --- Optuna-Steuerung ---
# n_trials = Anzahl NEUER Trials in DIESEM Lauf (nicht die Gesamtzahl der
# Study -- bei aktivem persistentem Storage kommen sie zu evtl. bereits
# vorhandenen Trials der Study HINZU, siehe Persistenz-Widgets unten).
# seed steuert den TPE-Sampler (nur bei einer NEU erstellten Study wirksam).
ot_w_n_trials = widgets.IntSlider(value=50, min=5, max=300, step=5, description="n_trials (NEUE Trials):", style={"description_width": "180px"}, layout=widgets.Layout(width="440px"))
ot_w_seed = widgets.IntText(value=42, description="Seed:", style={"description_width": "160px"})

# --- Persistenter SQLite-Storage + optuna-dashboard ---
# Checkbox AN (Default): tb.create_or_load_study() baut/laedt eine Study
# gegen eine SQLite-Datei -- ueberlebt Notebook-/Kernel-Neustarts, mehrere
# Laeufe mit demselben Study-Namen haengen Trials AN statt neu zu beginnen,
# UND waehrend das Tuning laeuft kann `optuna-dashboard sqlite:///<Pfad>
# --server wsgiref` (Windows -- siehe Abschnitt 5 oben) parallel gestartet
# werden, um den Fortschritt live im Browser zu verfolgen.
# Checkbox AUS: reine In-Memory-Study, kein Dashboard, kein Ueberleben eines
# Neustarts -- nur fuer schnelle Tests ohne Dashboard-Bedarf gedacht.
# Study-Name leer = Basket-Name wird verwendet (siehe ot-callback).
ot_w_persistent = widgets.Checkbox(value=True, description="Persistenter SQLite-Storage (fuer optuna-dashboard)", indent=False)
ot_w_study_name = widgets.Text(value="", placeholder="Default: Basket-Name", description="Study-Name:", style={"description_width": "160px"})
ot_w_storage_path = widgets.Text(value=str(tb.DEFAULT_STORAGE_PATH), description="SQLite-Datei:", style={"description_width": "160px"}, layout=widgets.Layout(width="560px"))

ot_run_button = widgets.Button(description="Optuna-Tuning starten", button_style="primary", icon="play")
ot_output = widgets.Output()
ot_last_result = {}  # haelt TuningResult (siehe tune_backtest.py) fuer Abschnitt 5b

def _ot_row(widget, description):
    """Baut eine Zeile: Widget links, kurze graue Erklaerung rechts daneben."""
    desc_html = widgets.HTML(
        f"<span style='color:#888; font-size:0.9em;'>{description}</span>",
        layout=widgets.Layout(margin="6px 0 0 12px"),
    )
    return widgets.HBox([widget, desc_html])


ot_ui = widgets.VBox([
    widgets.HTML("<b>Universum</b>"),
    _ot_row(ot_w_variant, "Cliquen-Cache-Variante (siehe Abschnitt 2): 'raw' oder 'residual'."),
    _ot_row(ot_w_basket, "Welcher TDA-Basket (Cluster-Gruppe) getunt wird."),

    widgets.HTML("<b>Zeitliche Trennung</b> (EIN Gesamtzeitraum -- Train/Val/Test werden automatisch berechnet, keine widerspruechliche manuelle Auswahl mehr moeglich)"),
    _ot_row(widgets.HBox([ot_w_overall_start, ot_w_overall_end]),
            "Gesamter Zeitraum, der geladen/verwendet wird."),
    _ot_row(ot_w_test_pct,
            "Wie viel Prozent des Gesamtzeitraums (am ENDE) als finaler, unberuehrter Testzeitraum reserviert werden."),
    _ot_row(ot_w_embargo_days,
            "Puffer (Tage) zwischen Train/Val Ende und Test Start -- verhindert Datenleck durch ueberlappende Lookback-Fenster."),
    widgets.HTML("<span style='color:#aaa; font-size:0.85em;'>Automatisch berechnet (nicht editierbar):</span>"),
    _ot_row(widgets.HBox([ot_w_train_val_start, ot_w_train_val_end]),
            "Zeitraum, INNERHALB dessen Optuna sucht (Walk-Forward-Fenster darin, siehe unten)."),
    _ot_row(widgets.HBox([ot_w_test_start, ot_w_test_end]),
            "Bleibt WAEHREND DES GESAMTEN TUNINGS unberuehrt, wird erst in Abschnitt 5b EINMAL final getestet."),

    widgets.HTML("<b>Walk-Forward-Fenster (innerhalb Train/Val)</b>"),
    _ot_row(ot_w_train_days, "Laenge des Trainingsfensters je Walk-Forward-Schritt (Tage)."),
    _ot_row(ot_w_val_days, "Laenge des Validierungsfensters je Walk-Forward-Schritt (Tage) -- hierauf wird jeder Trial bewertet."),
    _ot_row(ot_w_step_days, "Um wie viele Tage das Fenster bei jedem Walk-Forward-Schritt weiterrueckt."),
    _ot_row(ot_w_wf_embargo_days, "Puffer (Tage) zwischen Training und Validierung INNERHALB eines Walk-Forward-Schritts."),

    widgets.HTML("<b>Optuna</b>"),
    _ot_row(ot_w_n_trials, "Anzahl NEUER Trials in diesem Lauf (kommen bei aktiver Persistenz zu bestehenden Trials hinzu)."),
    _ot_row(ot_w_seed, "Zufalls-Seed fuer den TPE-Sampler -- nur bei einer NEU erstellten Study wirksam."),

    widgets.HTML("<b>Persistenz & Dashboard</b> (siehe Anleitung in Abschnitt 5 oben)"),
    _ot_row(ot_w_persistent, "An: Trials werden in einer SQLite-Datei gespeichert (ueberlebt Neustarts, mit optuna-dashboard live verfolgbar)."),
    _ot_row(ot_w_study_name, "Name der Optuna-Study. Leer = Basket-Name wird automatisch verwendet."),
    _ot_row(ot_w_storage_path, "Pfad zur SQLite-Datei, in der die Study gespeichert wird."),

    ot_run_button,
    ot_output,
])
display(ot_ui)


In [17]:
# =====================================================================
# Abschnitt 5: Callback -- startet einen Optuna-Tuning-Lauf ueber
# tune_backtest.tune_pairtrading_strategy() und zeigt die Ergebnisse an.
# =====================================================================
import matplotlib.pyplot as plt

def _ot_on_run_clicked(_button=None):
    with ot_output:
        clear_output(wait=True)

        # Pflichtfelder pruefen, BEVOR irgendein teurer Aufruf startet.
        for w, label in [
            (ot_w_train_val_start, "Train/Val Start"), (ot_w_train_val_end, "Train/Val Ende"),
            (ot_w_test_start, "Test Start"), (ot_w_test_end, "Test Ende"),
        ]:
            if w.value is None:
                print(f"Bitte '{label}' setzen.")
                return

        cliques_df = cliques_by_variant[ot_w_variant.value]
        cliques_df = cliques_df.loc[cliques_df["basket"] == ot_w_basket.value]

        # Study-Name-Default: Basket-Name, wenn das Textfeld leer gelassen wurde
        # (siehe ot-widgets) -- so hat jeder Basket standardmaessig seine eigene
        # Study, auch ohne dass explizit ein Name eingegeben werden muss.
        effective_study_name = ot_w_study_name.value or ot_w_basket.value
        print(f"Optuna-Tuning laeuft -- Basket: {ot_w_basket.value} | "
              f"Train/Val: {ot_w_train_val_start.value} -> {ot_w_train_val_end.value} | "
              f"Test (unberuehrt bis zum Schluss): {ot_w_test_start.value} -> {ot_w_test_end.value} | "
              f"n_trials={ot_w_n_trials.value} ...")
        if ot_w_persistent.value:
            print(f"Persistenter Storage: sqlite:///{ot_w_storage_path.value} | Study-Name: '{effective_study_name}' "
                  f"-- waehrend des Laufs mit optuna-dashboard live verfolgbar (siehe Anleitung oben).\n")
        else:
            print("Persistenter Storage: AUS (In-Memory, kein Dashboard).\n")

        # Der eigentliche Tuning-Lauf -- ALLE Logik (Suchraum, Stufe-1-Screen,
        # Stufe-2-Backtest, Zielgroesse, Walk-Forward, Persistenz) steckt in
        # tune_backtest.py, hier wird nur mit den Widget-Werten aufgerufen.
        # ValueError/RuntimeError werden gezielt abgefangen (siehe
        # tune_backtest.tune_pairtrading_strategy-Docstring): z.B. wenn die
        # Walk-Forward-Konfiguration kein einziges Fenster ergibt, oder wenn
        # kein Trial den Stufe-1-Screen uebersteht.
        try:
            result = tb.tune_pairtrading_strategy(
                basket=ot_w_basket.value, cliques_df=cliques_df, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
                timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
                train_val_start=ot_w_train_val_start.value, train_val_end=ot_w_train_val_end.value,
                test_start=ot_w_test_start.value, test_end=ot_w_test_end.value,
                train_days=ot_w_train_days.value, val_days=ot_w_val_days.value,
                step_days=ot_w_step_days.value, embargo_days=ot_w_wf_embargo_days.value,
                n_trials=ot_w_n_trials.value, seed=ot_w_seed.value,
                output_path=PROCESSED_DIR / "tuning" / f"optuna_trials_{ot_w_basket.value.replace(' ', '_')}.parquet",
                study_name=ot_w_study_name.value or None,
                storage_path=ot_w_storage_path.value,
                use_persistent_storage=ot_w_persistent.value,
            )
        except (ValueError, RuntimeError) as e:
            print(f"Tuning abgebrochen: {e}")
            return

        # fuer Abschnitt 5b (finale, einmalige Testzeitraum-Auswertung) merken.
        ot_last_result.clear()
        ot_last_result.update(result=result)

        tb.summarize_tuning_results([result])

        # --- Top-10-Tabelle aller abgeschlossenen (nicht gepruneten) Trials ---
        # "abgeschlossen" heisst: Stufe 1 bestanden UND voller Backtest gelaufen.
        # Gepruned Trials (Stufe 1 nicht bestanden) tauchen hier NICHT auf --
        # fuer sie wurde bewusst kein voller Backtest gerechnet, siehe Scatter-
        # Plot unten fuer eine vollstaendige Uebersicht inkl. gepruneter Trials.
        completed = result.trials_df.loc[result.trials_df["status"] == "completed"].sort_values("objective", ascending=False)
        if not completed.empty:
            print("\nTop 10 abgeschlossene Trials nach Objective:")
            display(completed.head(10)[[
                "lookback_candles", "p_content", "trade_mode", "recalib_every",
                "max_holding_hours", "max_concurrent_positions", "objective",
                "mean_fold_return", "std_fold_return", "stage1_ic",
            ]])

        # --- Scatter: Objective vs. Stufe-1-IC ueber ALLE Trials dieses Laufs ---
        # Zeigt visuell, ob der Stufe-1-Screen (IC-Schwelle SCREEN_MIN_IC,
        # gestrichelte Linie) tatsaechlich mit dem Stufe-2-Ergebnis (Objective)
        # korreliert -- ein guter Indikator, ob das Pruning-Kriterium sinnvoll
        # kalibriert ist (siehe tune_backtest.SCREEN_MIN_IC/SCREEN_MIN_SIGNALS).
        # Gepruned Trials werden bei y=0 als Kreuze dargestellt (sie haben kein
        # echtes Objective, da nie ein voller Backtest lief).
        fig, ax = plt.subplots(figsize=(6, 4.5))
        pruned_mask = result.trials_df["status"] == "pruned_stage1"
        ax.scatter(result.trials_df.loc[pruned_mask, "stage1_ic"], [0] * pruned_mask.sum(),
                   color="lightgrey", label=f"Stufe-1-gepruned (n={pruned_mask.sum()})", marker="x")
        ax.scatter(completed["stage1_ic"], completed["objective"], color="steelblue",
                   label=f"vollstaendig (n={len(completed)})")
        ax.axvline(tb.SCREEN_MIN_IC, color="grey", linestyle="--", linewidth=0.8, label=f"SCREEN_MIN_IC={tb.SCREEN_MIN_IC}")
        ax.set_xlabel("Stufe-1 Information Coefficient (IC)")
        ax.set_ylabel("Objective (0 = gepruned)")
        ax.set_title(f"Trials: Stufe-1-IC vs. finales Objective ({ot_w_basket.value})")
        ax.legend()
        plt.tight_layout()
        plt.show()


ot_run_button.on_click(_ot_on_run_clicked)


## 5b. Finale Testzeitraum-Auswertung (EINMALIG, nach Abschluss des Tunings)

Laeuft `run_pairtrading_backtest` mit den `best_params` aus Abschnitt 5 GENAU EINMAL auf dem
finalen Testzeitraum -- der bis zu diesem Punkt in KEINEM Trial angefasst wurde. Dies ist der
einzige Zeitpunkt, an dem der Testzeitraum ueberhaupt verwendet wird. Ergebnis hier ist die
Zahl, die in der Thesis als Out-of-Sample-Performance zitiert werden sollte -- nicht die
Validierungs-Objective-Werte aus Abschnitt 5 (die sind Teil der Modellwahl, nicht des Tests).


In [18]:
# =====================================================================
# Abschnitt 5b: EINMALIGE finale Auswertung auf dem unberuehrten Testzeitraum
# =====================================================================
# WICHTIG: diese Zelle sollte im gesamten Notebook nur EIN EINZIGES MAL nach
# Abschluss des Tunings ausgefuehrt werden. Sie verwendet best_params aus dem
# LETZTEN Optuna-Lauf (Abschnitt 5) und testet sie auf [Test Start, Test Ende)
# -- dem Zeitraum, der waehrend ALLER Trials in Abschnitt 5 nie angefasst
# wurde (siehe assert_no_test_period_leakage in pairtrading_engine.py).
# Das Ergebnis hier, NICHT die Validierungs-Objective-Werte aus Abschnitt 5,
# ist die Out-of-Sample-Performance-Zahl fuer die Thesis.
if not ot_last_result:
    print("Erst Abschnitt 5 (Optuna-Tuning) ausfuehren.")
else:
    result = ot_last_result["result"]
    best_params = result.best_params
    print(f"Finale Testzeitraum-Auswertung -- Basket: {result.basket} | "
          f"Testzeitraum: {ot_w_test_start.value} -> {ot_w_test_end.value} | "
          f"best_params={best_params}\n")

    cliques_df_test = cliques_by_variant[ot_w_variant.value]
    cliques_df_test = cliques_df_test.loc[cliques_df_test["basket"] == result.basket]

    # max_holding_hours (Optuna-Suchparameter, siehe Abschnitt 5) -> Kerzen
    # umrechnen, exakt dieselbe Konvention wie im Tuning-Callback
    # (5-Min-Kerzen: 60 Minuten / 5 = 12 Kerzen/Stunde).
    max_holding_candles_test = max(1, int(round(best_params["max_holding_hours"] * 60 / 5)))

    # run_pairtrading_backtest mit den getunten Signal-Parametern (best_params)
    # UND den fest vorgegebenen, nicht getunten Parametern aus tune_backtest.py
    # (FIXED_EWMA_DECAY, FIXED_CONFIDENCE, FIXED_FEE_BPS, FIXED_INITIAL_CAPITAL
    # -- siehe Modul-Docstring von tune_backtest.py, Abschnitt "Theoriegeleitete
    # Filterparameter vs. tunbare Signalparameter", fuer die Begruendung, warum
    # diese Werte fest sind statt Teil der Optuna-Suche zu sein).
    test_trades_df = pte.run_pairtrading_backtest(
        cliques_df=cliques_df_test, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
        timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
        start_date=ot_w_test_start.value, end_date=ot_w_test_end.value,
        decay=tb.FIXED_EWMA_DECAY, lookback_candles=best_params["lookback_candles"],
        p_content=best_params["p_content"], confidence=tb.FIXED_CONFIDENCE,
        max_holding_candles=max_holding_candles_test, fee_bps=tb.FIXED_FEE_BPS,
        trade_mode=best_params["trade_mode"], recalib_every=best_params["recalib_every"],
    )
    test_equity_curve = pte.compute_equity_curve(
        test_trades_df, initial_capital=tb.FIXED_INITIAL_CAPITAL,
        max_concurrent_positions=best_params["max_concurrent_positions"],
    )
    test_stats = pte.compute_backtest_stats(test_trades_df, test_equity_curve)

    if test_trades_df.empty:
        print("Keine Trades im finalen Testzeitraum mit den gewaehlten Parametern.")
    else:
        test_stats_display = pd.DataFrame([{
            "Anzahl Trades": test_stats["n_trades"],
            "Win-Rate": f"{test_stats['win_rate']:.1%}",
            "Sharpe": f"{test_stats['sharpe']:.2f}" if np.isfinite(test_stats["sharpe"]) else "n/a",
            "Max Drawdown": f"{test_stats['max_drawdown']:.2%}" if np.isfinite(test_stats["max_drawdown"]) else "n/a",
            "Gesamtrendite (finaler Test, out-of-sample)": f"{test_stats['total_return']:.2%}" if np.isfinite(test_stats["total_return"]) else "n/a",
        }]).T
        test_stats_display.columns = ["Wert"]
        display(test_stats_display)
        pte.plot_equity_curve(test_equity_curve, initial_capital=tb.FIXED_INITIAL_CAPITAL)


Erst Abschnitt 5 (Optuna-Tuning) ausfuehren.
